# Chapter 03 — If There's Any Doubt, It's Deterministic

**Companion to *Applied AI*.**

This notebook accompanies Chapter 3. The chapter states a governing rule and an
operational test. This notebook runs the seven operations the chapter opens
with and measures what happens when the deterministic ones are routed through a
stochastic component instead.

## Question

The chapter says six of its seven operations have exactly one correct answer.
**What does it actually cost to route those six through a sampler anyway?**

## What this notebook establishes

- All six deterministic operations, implemented, returning identical results on
  every one of many reruns.
- The same operations routed through a simulated sampler: measured variance
  across reruns, a token bill, and **failure modes that did not exist before**
  (an answer in the wrong format, a plausible wrong number).
- The generator/verifier asymmetry: a case where the solution cannot be written
  but the check can, run against many noisy candidates.

## What this notebook does **not** establish

- **The "model" here is a local random sampler, not a language model.** It is a
  stand-in that makes variance and format drift observable. No claim is made
  about how any real model would behave on these operations.
- Nothing here measures `temperature=0` nondeterminism. The chapter's evidence
  for that (1,000 completions, 80 unique, from He et al.) is a cited external
  measurement and is **not** reproduced here.
- Costs are illustrative arithmetic at a made-up token price.

## Setup

In [1]:
import hashlib
import random
import statistics
from dataclasses import dataclass

SEED = 42
random.seed(SEED)

PARAGRAPH = (
    "The new cache makes every page load 73% faster, according to the "
    "platform team [S1]. It stores rendered fragments close to readers."
)

BIBLIOGRAPHY = {"S1": {"year": 2026, "url": "https://example.invalid/s1"}}

def sha256(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

## The seven operations

The chapter's list. Six have exactly one correct answer; step 4 does not.

In [2]:
OPERATIONS = [
    (1, "Select the paragraph to review",                      "deterministic"),
    (2, "Decide whether it changed since the last review",     "deterministic"),
    (3, "Decide whether there is budget left for a call",      "deterministic"),
    (4, "Identify claims that need supporting evidence",       "STOCHASTIC"),
    (5, "Extract the flagged sentence and its position",       "deterministic"),
    (6, "Check a cited source resolves and its year matches",  "deterministic"),
    (7, "Decide whether to write the change to the file",      "deterministic"),
]

for n, desc, side in OPERATIONS:
    mark = "<-- needs a model" if side == "STOCHASTIC" else ""
    print(f"  {n}. {desc:<50} {side:<14}{mark}")

  1. Select the paragraph to review                     deterministic 
  2. Decide whether it changed since the last review    deterministic 
  3. Decide whether there is budget left for a call     deterministic 
  4. Identify claims that need supporting evidence      STOCHASTIC    <-- needs a model
  5. Extract the flagged sentence and its position      deterministic 
  6. Check a cited source resolves and its year matches deterministic 
  7. Decide whether to write the change to the file     deterministic 


## The six deterministic implementations

Each is computable from explicit state and rules — the chapter's current
operational test.

In [3]:
CORPUS = {"ch3-para-014": PARAGRAPH}

def op1_select(paragraph_id):                      # index lookup
    return CORPUS[paragraph_id]

def op2_changed(text, previous_sha):               # hash comparison
    return sha256(text) != previous_sha

def op3_budget(spend_so_far, cost_per_call, ceiling):   # arithmetic
    return (ceiling - spend_so_far) >= cost_per_call

def op5_extract(text, sentence):                   # parsing
    start = text.find(sentence)
    return None if start == -1 else (start, start + len(sentence))

def op6_source_ok(marker, claimed_year):           # table comparison
    entry = BIBLIOGRAPHY.get(marker)
    return entry is not None and entry["year"] == claimed_year

def op7_write(check_passed, has_authority):        # policy
    return "write" if (check_passed and has_authority) else "escalate"

flagged = "The new cache makes every page load 73% faster, according to the platform team [S1]."
prev_sha = sha256(PARAGRAPH)

results = {
    "op1 select":   op1_select("ch3-para-014")[:32] + "...",
    "op2 changed":  op2_changed(PARAGRAPH, prev_sha),
    "op3 budget":   op3_budget(0.40, 0.01, 0.50),
    "op5 extract":  op5_extract(PARAGRAPH, flagged),
    "op6 source":   op6_source_ok("S1", 2026),
    "op7 decision": op7_write(True, True),
}
for k, v in results.items():
    print(f"  {k:<14} {v}")

  op1 select     The new cache makes every page l...
  op2 changed    False
  op3 budget     True
  op5 extract    (0, 84)
  op6 source     True
  op7 decision   write


## Baseline: rerun them 500 times

In [4]:
RUNS = 500

def run_deterministic_once():
    return (
        op2_changed(PARAGRAPH, prev_sha),
        op3_budget(0.40, 0.01, 0.50),
        op5_extract(PARAGRAPH, flagged),
        op6_source_ok("S1", 2026),
    )

distinct = {run_deterministic_once() for _ in range(RUNS)}
print(f"{RUNS} reruns -> {len(distinct)} distinct result tuple(s)")
assert len(distinct) == 1, "deterministic operations must not vary"
print("assertion held: identical every time, at zero token cost")

500 reruns -> 1 distinct result tuple(s)
assertion held: identical every time, at zero token cost


## Experiment: route three of them through a sampler instead

The sampler stands in for a stochastic component. It is *mostly* right — which
is the point. A component that was usually wrong would never survive review.

In [5]:
PRICE_PER_1K = 1.00      # illustrative
TOKENS_PER_CALL = 420    # illustrative

@dataclass
class Sampler:
    """Local stand-in for a stochastic component. Not a language model."""
    accuracy: float = 0.94
    calls: int = 0
    tokens: int = 0

    def ask(self, kind, truth):
        self.calls += 1
        self.tokens += TOKENS_PER_CALL
        if random.random() < self.accuracy:
            return self._format(kind, truth)
        return self._wrong(kind, truth)

    def _format(self, kind, truth):
        if kind == "changed":
            return "Yes, it changed." if truth else "No, unchanged."
        if kind == "budget":
            return "There is budget remaining." if truth else "Budget exhausted."
        if kind == "span":
            return f"The sentence runs from character {truth[0]} to {truth[1]}."
        raise ValueError(kind)

    def _wrong(self, kind, truth):
        if kind == "changed":
            return "No, unchanged." if truth else "Yes, it changed."
        if kind == "budget":
            return "Budget exhausted." if truth else "There is budget remaining."
        if kind == "span":
            off = random.choice([-3, -1, 1, 4])
            return f"The sentence runs from character {truth[0] + off} to {truth[1] + off}."
        raise ValueError(kind)

    @property
    def cost(self):
        return self.tokens / 1000 * PRICE_PER_1K

def parse_bool(prose):
    low = prose.lower()
    if low.startswith("yes") or "remaining" in low:
        return True
    if low.startswith("no") or "exhausted" in low:
        return False
    raise ValueError(f"unparseable: {prose!r}")   # a new failure mode

def parse_span(prose):
    import re
    m = re.search(r"character (\d+) to (\d+)", prose)
    if not m:
        raise ValueError(f"unparseable: {prose!r}")
    return (int(m.group(1)), int(m.group(2)))

In [6]:
truth = run_deterministic_once()
truth_changed, truth_budget, truth_span, _ = truth

sampler = Sampler()
observed, unparseable = [], 0

for _ in range(RUNS):
    try:
        observed.append((
            parse_bool(sampler.ask("changed", truth_changed)),
            parse_bool(sampler.ask("budget", truth_budget)),
            parse_span(sampler.ask("span", truth_span)),
        ))
    except ValueError:
        # A third outcome that did not exist before: the answer arrived and
        # could not be read at all. We count it rather than crashing.
        unparseable += 1

distinct_s = {o for o in observed}
correct = sum(1 for o in observed if o == (truth_changed, truth_budget, truth_span))

print(f"{RUNS} reruns -> {len(distinct_s)} distinct result tuple(s)")
print(f"exactly correct on all three : {correct}/{RUNS} = {correct/RUNS:.1%}")
print(f"wrong but well-formed        : {len(observed) - correct}/{RUNS}")
print(f"unparseable (new failure mode): {unparseable}/{RUNS}")
print(f"model calls: {sampler.calls}   tokens: {sampler.tokens:,}   cost: ${sampler.cost:,.2f}")

500 reruns -> 7 distinct result tuple(s)
exactly correct on all three : 414/500 = 82.8%
wrong but well-formed        : 67/500
unparseable (new failure mode): 19/500
model calls: 1500   tokens: 630,000   cost: $630.00


## Observation

Put the two side by side.

In [7]:
rows = [
    ("distinct results over 500 reruns", "1", f"{len(distinct_s)}"),
    ("fully correct runs", f"{RUNS}/{RUNS} (100.0%)", f"{correct}/{RUNS} ({correct/RUNS:.1%})"),
    ("model calls", "0", f"{sampler.calls:,}"),
    ("token cost", "$0.00", f"${sampler.cost:,.2f}"),
    ("unparseable answers", "0", f"{unparseable}"),
    ("wrong answer is distinguishable from right", "n/a", "no"),
]
print(f"{'':<44}{'deterministic':<18}{'routed through sampler'}")
print("-" * 84)
for label, a, b in rows:
    print(f"{label:<44}{a:<18}{b}")

                                            deterministic     routed through sampler
------------------------------------------------------------------------------------
distinct results over 500 reruns            1                 7
fully correct runs                          500/500 (100.0%)  414/500 (82.8%)
model calls                                 0                 1,500
token cost                                  $0.00             $630.00
unparseable answers                         0                 19
wrong answer is distinguishable from right  n/a               no


The third row of the sampler column is the one worth sitting with. Its wrong
answers are **the same shape as its right answers**: a plausible sentence with
a plausible number in it. Nothing downstream can tell them apart, which is why
the chapter insists the annoyance test is a design test and not a preference.

> **If you would be annoyed to get a different answer when you re-run it, it
> should not be a model call.**

## The asymmetry that makes any of this possible

Step 4 genuinely needs a proposal from a space you cannot economically
enumerate. The chapter's answer is not to make it deterministic — it is to put
a **deterministic verifier** behind it.

Here the "generator" proposes flagged sentences, badly. The verifier is exact
and free: does the flagged span appear **verbatim** in the paragraph?

In [8]:
def bad_generator(text, rng):
    """Proposes a flagged sentence. Sometimes paraphrases, which is the defect."""
    real = flagged
    r = rng.random()
    if r < 0.55:
        return real                                   # quotes correctly
    if r < 0.80:
        return real.replace("73%", "seventy-three percent")   # paraphrase
    return "The cache is fast, the team says [S1]."           # invention

def verifier(text, proposal):
    """Deterministic, exact, free. Chapter 3's cheap check."""
    return text.find(proposal) != -1

rng = random.Random(SEED)
N = 400
proposals = [bad_generator(PARAGRAPH, rng) for _ in range(N)]
accepted = [p for p in proposals if verifier(PARAGRAPH, p)]

print(f"candidates generated : {N}")
print(f"accepted by verifier : {len(accepted)}  ({len(accepted)/N:.1%})")
print(f"rejected             : {N - len(accepted)}  ({1 - len(accepted)/N:.1%})")
print()
print("every accepted candidate is verbatim in the paragraph:")
assert all(PARAGRAPH.find(p) != -1 for p in accepted)
print("  assertion held")
print()
print("distinct accepted strings:", len(set(accepted)))

candidates generated : 400
accepted by verifier : 217  (54.2%)
rejected             : 183  (45.8%)

every accepted candidate is verbatim in the paragraph:
  assertion held

distinct accepted strings: 1


## Interpretation

The verifier contributed all of the usable correctness and cost nothing to run.
The generator contributed **variance** — proposals from a space nobody
enumerated — and a large fraction of those proposals were unusable.

That is the architecture the chapter argues for:

```text
deterministic  ->  stochastic proposal  ->  deterministic verification
```

Two boundaries this notebook deliberately does not cross:

- **A passing check is not correctness.** The verifier here establishes that a
  quoted span exists, and nothing else. Chapter 21 shows a well-built check
  accepting a grounded, verbatim, *wrong* answer.
- **A cheap check is not always available.** Where verification is delayed,
  subjective or adversarial, this architecture does not apply. Chapter 4's
  fraud-detection row is the case that looks perfect and is not.

The chapter's override also stands: when the specification cost genuinely
exceeds the value of determinism, use a model — and record that you chose to.

## Try it yourself

1. **Raise the sampler's accuracy to 0.999** and re-run. The variance shrinks
   and never reaches zero, and the bill stays. At what accuracy would you
   accept it for the budget arithmetic — and would you accept it for the
   *bank's* budget arithmetic?
2. **Delete the verifier.** Accept every candidate from `bad_generator` and look
   at what enters the record. This is the chapter's "demo, not a process".
3. **Weaken the verifier** so it only checks that the proposal mentions `[S1]`.
   Watch the invented sentence pass. Then read Chapter 21.
4. **Make the parse fail.** Have the sampler occasionally answer `"It depends."`
   and see which of the two architectures notices.